# Time-aware validation
I build only features available before the prediction timestamp and evaluate them with expanding chronological folds.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error

rng=np.random.default_rng(7); n=1200; t=np.arange(n)
y=30+.02*t+8*np.sin(2*np.pi*t/7)+rng.normal(0,2.5,n)
df=pd.DataFrame({'y':y})
for lag in [1,2,7,14,28]: df[f'lag_{lag}']=df.y.shift(lag)
df['roll7_mean']=df.y.shift(1).rolling(7).mean()
df['roll28_std']=df.y.shift(1).rolling(28).std()
df=df.dropna()
X=df.drop(columns='y'); target=df.y
cv=TimeSeriesSplit(n_splits=5)
m=HistGradientBoostingRegressor(max_depth=4,learning_rate=.06,random_state=42)
mae=-cross_val_score(m,X,target,cv=cv,scoring='neg_mean_absolute_error')
pd.Series(mae,index=[f'fold_{i+1}' for i in range(len(mae))],name='MAE').round(3)


In [ ]:
split=-150
m.fit(X.iloc[:split],target.iloc[:split])
pred=m.predict(X.iloc[split:])
print('final chronological MAE:',round(mean_absolute_error(target.iloc[split:],pred),3))


## Leakage rule
A centered rolling mean or a random train/test split would use future context. In forecasting, feature availability is part of the feature definition; a numerically correct column can still be impossible at prediction time.